# 04 — Partition Evolution

Iceberg lets you change the partition layout without rewriting historical data.
Old files keep their original spec; new writes use the new spec; reads transparently
stitch them together.

We'll go from `days(ts)` (set up in 01) to `hours(ts)`, write a new batch under the new
spec, and verify both partitions coexist.

**Prerequisites:** Run `03_schema_evolution.ipynb` first.

In [1]:
import datetime

import polars as pl
from pyiceberg.io.pyarrow import schema_to_pyarrow
from pyiceberg.transforms import HourTransform

import os

from pyiceberg.catalog.rest import RestCatalog

NESSIE_URI       = os.environ["NESSIE_URI"]
S3_ENDPOINT      = os.environ["AWS_S3_ENDPOINT"]
S3_ACCESS_KEY    = os.environ["AWS_ACCESS_KEY_ID"]
S3_SECRET_KEY    = os.environ["AWS_SECRET_ACCESS_KEY"]
WAREHOUSE_BUCKET = os.environ["ICEBERG_WAREHOUSE_BUCKET"]
WAREHOUSE_URI    = f"s3://{WAREHOUSE_BUCKET}/warehouse"

catalog = RestCatalog(
    name="nessie",
    **{
        "uri": NESSIE_URI,
        "warehouse": WAREHOUSE_URI,
        "s3.endpoint": S3_ENDPOINT,
        "s3.access-key-id": S3_ACCESS_KEY,
        "s3.secret-access-key": S3_SECRET_KEY,
        "s3.path-style-access": "true",
        "s3.region": "us-east-1",
    },
)

print("Connected to catalog:", catalog.name)

table = catalog.load_table(("demo", "events"))
table.refresh()
print("Current partition spec:", table.spec())

Connected to catalog: nessie
Current partition spec: [
  1000: ts_day: day(5)
]


## 1. Inspect existing partitions

In [2]:
partitions = pl.from_arrow(table.inspect.partitions())
print(f"Partition rows: {len(partitions)}")
partitions

Partition rows: 4


partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
struct[1],i32,i64,i32,i64,i64,i32,i64,i32,datetime[ms],i64
{2024-01-15},0,2,2,4925,0,0,0,0,2026-05-15 11:06:26.544,4874759031295520682
{2024-01-18},0,2,1,2516,0,0,0,0,null,null
{2024-01-16},0,2,1,2521,0,0,0,0,null,null
{2024-01-17},0,1,1,2459,0,0,0,0,null,null


## 2. Evolve the partition spec — replace `days(ts)` with `hours(ts)`

Note: in PyIceberg you can't delete and add a field on the same source
column in a single transaction commit if the name collides. We drop the old
field first, commit, then add the new one.

In [3]:
with table.update_spec() as us:
    us.remove_field("ts_day")
    us.add_field("ts", HourTransform(), "ts_hour")

table.refresh()
print("New partition spec:", table.spec())

New partition spec: [
  1001: ts_hour: hour(5)
]


## 3. Write a new batch under the new spec

In [4]:
UTC = datetime.timezone.utc

df_new = pl.DataFrame(
    {
        "event_id":   ["e100", "e101", "e102"],
        "user_id":    [201, 201, 202],
        "event_kind": ["click", "view", "click"],
        "amount":     [0.0, 0.0, 0.0],
        "ts": [
            datetime.datetime(2024, 2, 1, 9,  0, 0, tzinfo=UTC),
            datetime.datetime(2024, 2, 1, 9, 30, 0, tzinfo=UTC),
            datetime.datetime(2024, 2, 1, 10, 0, 0, tzinfo=UTC),
        ],
    }
)

# Column order may differ from the (possibly evolved) table schema —
# reorder to match, then cast nullability/types to align with Iceberg.
field_names = [f.name for f in table.schema().fields]
table.append(
    df_new.select(field_names).to_arrow().cast(schema_to_pyarrow(table.schema()))
)
table.refresh()
print("Snapshots:", len(table.history()))

Snapshots: 1


## 4. Inspect partitions — both specs should appear

In [5]:
partitions = pl.from_arrow(table.inspect.partitions())
print(f"Partition rows after evolution: {len(partitions)}")
partitions

Partition rows after evolution: 6


partition,spec_id,record_count,file_count,total_data_file_size_in_bytes,position_delete_record_count,position_delete_file_count,equality_delete_record_count,equality_delete_file_count,last_updated_at,last_updated_snapshot_id
struct[2],i32,i64,i32,i64,i64,i32,i64,i32,datetime[ms],i64
"{null,474105}",1,2,1,2489,0,0,0,0,2026-05-15 11:06:33.022,7176660290452338699
"{null,474106}",1,1,1,2466,0,0,0,0,2026-05-15 11:06:33.022,7176660290452338699
"{2024-01-15,null}",0,2,2,4925,0,0,0,0,null,null
"{2024-01-18,null}",0,2,1,2516,0,0,0,0,null,null
"{2024-01-16,null}",0,2,1,2521,0,0,0,0,null,null
"{2024-01-17,null}",0,1,1,2459,0,0,0,0,null,null


## 5. Read across both specs

Iceberg handles the spec mismatch transparently — one logical scan covers both layouts.

In [6]:
df = pl.from_arrow(table.scan().to_arrow())
print(f"Total rows: {len(df)}")
df.sort("ts")

Total rows: 10


ts,event_id,user_id,event_kind,amount
"datetime[μs, UTC]",str,i64,str,f64
2024-01-15 10:00:00 UTC,"""e001""",101,"""click""",0.0
2024-01-15 11:00:00 UTC,"""e002""",102,"""view""",5.99
2024-01-16 09:00:00 UTC,"""e003""",103,"""click""",0.0
2024-01-16 14:00:00 UTC,"""e004""",101,"""purchase""",49.99
2024-01-17 08:00:00 UTC,"""e005""",104,"""view""",0.0
2024-01-18 09:00:00 UTC,"""e006""",105,"""click""",0.0
2024-01-18 10:30:00 UTC,"""e007""",101,"""purchase""",99.0
2024-02-01 09:00:00 UTC,"""e100""",201,"""click""",0.0
2024-02-01 09:30:00 UTC,"""e101""",201,"""view""",0.0


---
**Next:** [05_row_mutations.ipynb](05_row_mutations.ipynb) — upserts, deletes, overwrites.